In [1]:
import os
import pandas as pd
import numpy as np
from scipy.ndimage import gaussian_filter1d
from scipy import stats

In [2]:
# Read in the smoothed data produced by 01__data_smoothing.ipynb
open_path = '../data/data_gen/open_smoothed.pkl'
curfew_path = '../data/data_gen/curfew_smoothed.pkl'

df_open_smoothed = pd.read_pickle(open_path)
df_curfew_smoothed = pd.read_pickle(curfew_path)

In [3]:
# Exclude extraordinary days before building the averages.
# - open: 23rd October
# - curfew: 12th December (a Saturday that was treated as a workday - excluded rather than reclassified)
df_open_smoothed = df_open_smoothed[~((df_open_smoothed['month'] == 10) & (df_open_smoothed['day'] == 23))].reset_index(drop=True)
df_curfew_smoothed = df_curfew_smoothed[~((df_curfew_smoothed['month'] == 12) & (df_curfew_smoothed['day'] == 12))].reset_index(drop=True)

In [4]:
# create datetime information as preparation for data aggregation
# first differentiate between weekday and weekend

def add_datetime_cols(df):
    """Create 'date', 'weekday' (True for weekdays) and 'hour' columns from year/month/day/hour."""
    # Convert float values to integers before converting to strings
    date_components = df[['year', 'month', 'day', 'hour']].astype(int)

    # Concatenate the integer components into a single string
    date_string = date_components.apply(lambda x: ' '.join(x.astype(str)), axis=1)

    # Convert the concatenated string to datetime format
    df.loc[:, 'date'] = pd.to_datetime(date_string, format='%Y %m %d %H')

    # create weekday/weekend column True for weekdays
    df.loc[:, 'weekday'] = df['date'].dt.weekday < 5

    # create hour column
    df.loc[:, 'hour'] = df['date'].dt.hour

    return df

df_open_smoothed = add_datetime_cols(df_open_smoothed)
df_curfew_smoothed = add_datetime_cols(df_curfew_smoothed)

##### Weekly presence masking

A raster_id's loc_work / loc_home is set to NA for its ENTIRE history if it does NOT have
work / anyone home in EVERY week of the period 

In [5]:
def apply_weekly_mask(df):
    """Create loc_work_weekly / loc_home_weekly: NA for the whole raster if it does NOT have
    work / anyone home in EVERY week of the period"""
    df = df.copy()
    df['week'] = df['date'].dt.isocalendar().week
    df['year_iso'] = df['date'].dt.isocalendar().year

    weekly_presence = df.groupby(['raster_id', 'year_iso', 'week']).agg(
        max_loc_work=('loc_work', 'max'),
        max_loc_home=('loc_home', 'max')
    ).reset_index()
    weekly_presence['work_flag'] = weekly_presence['max_loc_work'] >= 1
    weekly_presence['home_flag'] = weekly_presence['max_loc_home'] >= 1

    # Raster-level: does this raster have work / home activity in EVERY week it appears in?
    work_every_week = weekly_presence.groupby('raster_id')['work_flag'].all()
    home_every_week = weekly_presence.groupby('raster_id')['home_flag'].all()

    rasters_with_work_every_week = set(work_every_week[work_every_week].index)
    rasters_with_home_every_week = set(home_every_week[home_every_week].index)

    df['loc_work_weekly'] = df['loc_work'].copy()
    df.loc[~df['raster_id'].isin(rasters_with_work_every_week), 'loc_work_weekly'] = np.nan

    df['loc_home_weekly'] = df['loc_home'].copy()
    df.loc[~df['raster_id'].isin(rasters_with_home_every_week), 'loc_home_weekly'] = np.nan

    df.drop(columns=['week', 'year_iso'], inplace=True)
    return df

df_open_smoothed = apply_weekly_mask(df_open_smoothed)
df_curfew_smoothed = apply_weekly_mask(df_curfew_smoothed)

In [6]:
os.makedirs("../output/data", exist_ok=True)

df_open_smoothed.to_pickle("../data/data_gen/open_smoothed_2month.pkl")
df_curfew_smoothed.to_pickle("../data/data_gen/curfew_smoothed_cleaned.pkl")

### Create average 48 hour sequence

###### Now I created all the necessary information on work/home locations and times.

In [7]:
def build_48hr_average(df):
    """Collapse hourly data into a 48hr sequence per raster_id: hours 0-23 for weekdays,
    24-47 for weekends, take the mean across all matching weekday/weekend hours."""
    # group data by daytype and hour and calculate the mean and reset index
    avg_day = df.groupby(['raster_id', 'weekday', 'hour']).mean().reset_index()

    # drop all non-necessary columns
    avg_day.drop(columns=[c for c in ['year', 'month', 'day'] if c in avg_day.columns], inplace=True)

    # create a sequence of 48 hrs: hour_reordered runs 0-23 (weekday) then 24-47 (weekend)
    avg_day['hour_reordered'] = avg_day['hour']
    avg_day.loc[~avg_day['weekday'], 'hour_reordered'] += 24

    avg_day.sort_values(by=['weekday', 'hour_reordered'], inplace=True)
    avg_day.reset_index(drop=True, inplace=True)
    avg_day.drop(columns=['weekday', 'hour'], inplace=True)

    return avg_day

df_open_smooth_avg_day = build_48hr_average(df_open_smoothed)
df_curfew_smooth_avg_day = build_48hr_average(df_curfew_smoothed)

### Temporal smoothing

In [8]:
# Function to apply Gaussian filter to a specified column and add the results as a new column
def apply_gaussian_filter(group, input_col, output_col, sigma=1):
    group[output_col] = gaussian_filter1d(group[input_col], sigma=sigma)
    return group

In [ ]:
def smooth_temporal(df):
    """Gaussian-smooth loc_home_weekly, loc_work_weekly and traffic (via a +1 pseudocount to
    avoid log(0), producing loc_home_gaussian / loc_work_gaussian / traffic_gaussian."""
    df['loc_home_no0'] = df['loc_home_weekly'] + 1
    df['loc_work_no0'] = df['loc_work_weekly'] + 1
    df['traffic_no0'] = df['traffic'] + 1

    # Apply the function to each group
    df = df.groupby('raster_id').apply(apply_gaussian_filter, input_col='loc_home_no0', output_col='loc_home_gaussian').reset_index(drop=True)
    df = df.groupby('raster_id').apply(apply_gaussian_filter, input_col='loc_work_no0', output_col='loc_work_gaussian').reset_index(drop=True)
    df = df.groupby('raster_id').apply(apply_gaussian_filter, input_col='traffic_no0', output_col='traffic_gaussian').reset_index(drop=True)

    return df

df_open_smooth_avg_day = smooth_temporal(df_open_smooth_avg_day)
df_curfew_smooth_avg_day = smooth_temporal(df_curfew_smooth_avg_day)

In [ ]:
def add_log_zscore_norm(df):
    """Log-transform the Gaussian-smoothed work/traffic sequences, then z-score each location's
    values across its own 48hr sequence. Adds log_work_no0, log_traffic_no0, log_work_gauss_norm, log_traffic_gauss_norm.
    Returns the updated df plus the log_work_no0 pivot table (raw log) and its z-scored version."""
    df = df.copy()
    df['log_work_no0'] = np.log(df['loc_work_gaussian'])
    df['log_traffic_no0'] = np.log(df['traffic_gaussian'])

    work_matrix = df.pivot_table(index='raster_id', columns='hour_reordered', values='log_work_no0', aggfunc='first')
    traffic_matrix = df.pivot_table(index='raster_id', columns='hour_reordered', values='log_traffic_no0', aggfunc='first')

    z_work = stats.zscore(work_matrix, axis=1)
    z_traffic = stats.zscore(traffic_matrix, axis=1)

    z_work_pivoted = z_work.reset_index().melt(id_vars='raster_id', var_name='hour_reordered', value_name='log_work_gauss_norm')
    z_traffic_pivoted = z_traffic.reset_index().melt(id_vars='raster_id', var_name='hour_reordered', value_name='log_traffic_gauss_norm')

    df = df.merge(z_work_pivoted, on=['raster_id', 'hour_reordered'], how='left')
    df = df.merge(z_traffic_pivoted, on=['raster_id', 'hour_reordered'], how='left')

    return df, work_matrix, z_work

df_open_smooth_avg_day, work_matrix_gaus, z_work_gaus = add_log_zscore_norm(df_open_smooth_avg_day)
df_curfew_smooth_avg_day, _, _ = add_log_zscore_norm(df_curfew_smooth_avg_day)

In [ ]:
# Save the 48hr average sequences
os.makedirs("../output/data/48hr_averages", exist_ok=True)

df_open_smooth_avg_day.to_pickle("../output/data/48hr_averages/open_avg48hrs.pkl")
df_curfew_smooth_avg_day.to_pickle("../output/data/48hr_averages/curfew_avg48hrs.pkl")

### Clustering

In [13]:
os.makedirs("../output/data/matrices", exist_ok=True)

# Raw log matrix
work_matrix_gaus.to_pickle("../output/data/matrices/work_matrix_weekly_gaussian.pkl")

# z-scored/normalized matrix
z_work_gaus.to_pickle("../output/data/matrices/z_work_weekly_gaussian.pkl")